# Dermatological multimodal dataset · Part 2: Privacy and k-anonymity
**Author:** Marco Larriba Rius

In [ ]:
import pandas as pd, numpy as np, hashlib, os, glob, warnings
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=0.9)
plt.rcParams.update({'figure.dpi':130,'axes.spines.top':False,'axes.spines.right':False})

# Locate the local _data folder robustly (works regardless of the kernel's working directory).
def _locate_data(name='_data'):
    p = Path.cwd()
    for base in [p, *p.parents][:6]:
        if (base/name/'images').is_dir() or (base/name/'metadata_skin_cancer.csv').exists():
            return base/name
    for hit in glob.glob(str(p/'**'/name/'metadata_skin_cancer.csv'), recursive=True):
        return Path(hit).parent
    return Path(name)
DATA_DIR    = _locate_data()
META_PATH   = str(DATA_DIR/'metadata_skin_cancer.csv')
IMG_DIR     = str(DATA_DIR/'images')
FIGURES_DIR = DATA_DIR.parent/'figures'; FIGURES_DIR.mkdir(exist_ok=True)
df = pd.read_csv(META_PATH); print('Shape:', df.shape)

# Images are loaded LOCALLY from _data/images (unzip archive.zip there). Never Colab/kagglehub.
img_files  = glob.glob(os.path.join(IMG_DIR, '**', '*.png'), recursive=True)
RUN_IMAGES = len(img_files) > 0
print(f'DATA_DIR = {DATA_DIR}  ->  RUN_IMAGES = {RUN_IMAGES} ({len(img_files)} PNGs)')

## Step 2 - Privacy risk assessment

In [ ]:
# Classify every attribute by its power to identify a patient (assessment only, no transformation).
#   direct      = single out a person on their own        (removed in Step 3.1)
#   quasi       = re-identify in combination               (generalised in Step 3.2)
#   sensitive   = the medical payload, kept granular       (special_cat = GDPR Art. 9)
#   normal      = neither identifying nor sensitive
# The images are direct identifiers too (faces / body parts) and are excluded from the release.
direct      = ['dni', 'patient_id', 'lesion_id', 'img_id']
quasi       = ['age', 'gender', 'province', 'city', 'postal_code', 'hospital_id',
               'background_father', 'background_mother']      # ancestry: linkable origin + special cat.
sensitive   = ['diagnostic', 'biopsed', 'region', 'itch', 'grew', 'hurt', 'changed', 'bleed',
               'elevation', 'fitspatrick', 'diameter_1', 'diameter_2', 'smoke', 'drink', 'pesticide',
               'skin_cancer_history', 'cancer_history', 'background_father', 'background_mother']
normal      = ['has_piped_water', 'has_sewage_system']
special_cat = ['diagnostic', 'biopsed', 'region', 'background_father', 'background_mother']  # GDPR Art. 9
# Report cardinality and missing rate per tier (evidence for the classification).
for c in direct: print(f'DIRECT      {c:18s} unique={df[c].nunique():5d}')
print()
for c in quasi:  print(f'QUASI       {c:18s} unique={df[c].nunique():5d}  missing={df[c].isna().mean()*100:3.0f}%')
print()
print('SENSITIVE   :', len(sensitive), 'cols | special category (Art.9):', special_cat)
print('NORMAL      :', normal)

In [ ]:
# Linkage risk: how many patients are unique on the raw quasi-identifiers (one row per patient).
dp = df.drop_duplicates('patient_id').copy()
sizes = dp.groupby([dp[q].astype(str) for q in quasi]).size()   # equivalence-class size per patient
print(f'Patients: {len(dp)} | unique QI combinations: {len(sizes)} | '
      f'singletons: {int((sizes==1).sum())} ({(sizes==1).sum()/len(dp)*100:.0f}% uniquely identifiable)')

In [ ]:
# EXIF inspection (capture time / GPS / device). Optional here: images are excluded from any release.
if RUN_IMAGES:
    from PIL import Image
    from PIL.ExifTags import TAGS
    SENSITIVE_TAGS = {'DateTime', 'DateTimeOriginal', 'GPSInfo', 'Make', 'Model'}
    paths = glob.glob(os.path.join(IMG_DIR, '**', '*.png'), recursive=True)
    found = [{TAGS.get(k, k): v for k, v in Image.open(p).getexif().items()}
             for p in paths if Image.open(p).getexif()]
    print(f'Images scanned: {len(paths)} | with EXIF: {len(found)}')
    print('Privacy-relevant tags present:', {tg for t in found for tg in t if tg in SENSITIVE_TAGS} or 'none')
else:
    print('SKIPPED (no local images).')

## Step 3 - Anonymisation process

In [ ]:
# Step 3.A - Data-quality cleaning: build the polished table the privacy process runs on.
BOOL=['smoke','drink','pesticide','skin_cancer_history','cancer_history','has_piped_water',
      'has_sewage_system','itch','grew','hurt','changed','bleed','elevation','biopsed']
def to01(s):                                                   # unify mixed True/False, TRUE/FALSE, UNK
    return s.astype(str).str.strip().str.upper().map({'TRUE':1,'FALSE':0,'YES':1,'NO':0})
g = dp.copy()
for c in BOOL:                                                 # booleans -> {0,1}; unmapped -> not_assessed
    m=to01(g[c]); g[c]=m.where(m.notna(),'not_assessed')
for c in ['gender','background_father','background_mother']:   # MNAR categorical block -> not_assessed
    g[c]=g[c].where(g[c].notna(),'not_assessed')
g['fitspatrick']=pd.to_numeric(g['fitspatrick'],errors='coerce')   # ordinal skin-type scale
g['clinical_assessed']=g['diameter_1'].notna().astype(int)         # 1 = lesion clinically worked up
g=g.drop(columns=['diameter_2'])                                   # redundant (r=0.89 with diameter_1)
print('Cleaned table:', g.shape, '| residual nulls in QIs ->', {q:int(g[q].isna().sum()) for q in quasi})

In [ ]:
# Step 3.1 - Remove direct identifiers; pseudonymise patient_id.
#   tokenisation: patient_id -> artificial token; vault: real<->token map kept SEPARATELY (reversible).
import secrets
ids       = g['patient_id'].astype(str).unique()
token_map = {pid: 'PAT_' + secrets.token_hex(3).upper() for pid in ids}
vault     = pd.DataFrame({'patient_id': list(token_map), 'pid_token': list(token_map.values())})
g['pid_token'] = g['patient_id'].astype(str).map(token_map)
release = g.drop(columns=['dni', 'lesion_id', 'img_id', 'patient_id'])   # raw identifiers leave the data
print('Pseudonymised %d patients | vault rows (kept separately): %d | e.g. %s -> %s'
      % (len(ids), len(vault), vault.iloc[0]['patient_id'], vault.iloc[0]['pid_token']))
print('Released table carries no direct identifiers. Columns:', release.shape[1])

In [ ]:
# Step 3.1b - Masking demo: REDACTION on dni keeps only the trailing characters and masks the rest.
def redact(s): s=str(s); return '*'*(len(s)-3)+s[-3:]
demo = df['dni'].dropna().head(3)
print('Redaction (dni):', list(zip(demo.tolist(), demo.map(redact).tolist())))

In [ ]:
# Step 3.2 - Generalise the quasi-identifiers (all kept in the process, none dropped):
#   age          -> 10-year bands
#   ancestry     -> one macro-region (father, else mother)
#   geography    -> postal_code/city rolled up to province (province = single geographic QI)
#   hospital_id  -> its province
MACRO={'POMERANIA':'NC-European','GERMANY':'NC-European','POLAND':'NC-European','CZECH':'NC-European',
       'AUSTRIA':'NC-European','NETHERLANDS':'NC-European','NORWAY':'NC-European','FRANCE':'NC-European',
       'ITALY':'S-European','PORTUGAL':'S-European','SPAIN':'S-European','BRAZIL':'Brazilian','BRASIL':'Brazilian'}
def macro(v):
    v=str(v).strip().upper()
    if v in ('NAN','NOT_ASSESSED','UNK',''): return 'not_assessed'
    return MACRO.get(v,'Other')
fa=g['background_father'].map(macro); mo=g['background_mother'].map(macro)
g['ancestry_macro']=np.where(fa!='not_assessed', fa, mo)              # father's origin, else mother's
g['age_band']  = pd.cut(df.loc[g.index,'age'], range(0,101,10)).astype(str)
g['hosp_prov'] = g['hospital_id'].astype(str).str.extract(r'HOSP-([A-Z]+)-')[0]   # province code in the id
city2prov   = df.loc[g.index].dropna(subset=['city']).groupby('city')['province'].first()
postal_zone = df.loc[g.index,'postal_code'].astype('Int64').astype(str).str.zfill(5).str[:2]  # 2-digit = province
gen_qi = ['age_band', 'gender', 'province', 'hosp_prov', 'ancestry_macro']   # QIs entering the k-anonymity
print('city -> province: %d cities roll up to %d provinces' % (g['city'].nunique(), g['city'].map(city2prov).nunique()))
print('postal_code -> province: %d codes roll up to %d zones' % (g['postal_code'].nunique(), postal_zone.nunique()))
raw_sizes = dp.groupby([dp[q].astype(str) for q in quasi]).size()
gen_sizes = g.groupby([g[q].astype(str) for q in gen_qi]).size()
print('Raw QIs        : singletons=%d  classes=%d'%(int((raw_sizes==1).sum()), len(raw_sizes)))
print('Generalised QIs: singletons=%d  classes=%d'%(int((gen_sizes==1).sum()), len(gen_sizes)))
print('ancestry_macro :', dict(g['ancestry_macro'].value_counts()))

In [ ]:
# Step 3.2b - Age generalisation hierarchy (L0 exact .. L4 category): % patients retained at each
# level for several k (other QIs fixed). Shows how coarser bands trade utility for privacy.
def retained(d, qi, k):
    s=d.groupby([d[c].astype(str) for c in qi])[qi[0]].transform('size'); return round((s>=k).mean()*100,1)
def with_age(col):
    gg=g.copy(); gg['age_band']=col; return gg                  # swap the age band, keep the other QIs
hier={'L0 exact':  df.loc[g.index,'age'].astype(str),
      'L1 5y':     pd.cut(df.loc[g.index,'age'], range(0,101,5)).astype(str),
      'L2 10y':    pd.cut(df.loc[g.index,'age'], range(0,101,10)).astype(str),
      'L3 20y':    pd.cut(df.loc[g.index,'age'], range(0,101,20)).astype(str),
      'L4 category':pd.cut(df.loc[g.index,'age'], [0,40,150], labels=['<40','40+']).astype(str)}
rows=[{'level':n,'k=2':retained(with_age(c),gen_qi,2),'k=3':retained(with_age(c),gen_qi,3),
       'k=5':retained(with_age(c),gen_qi,5)} for n,c in hier.items()]
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Step 3.3 - Null check before suppression: every QI must be complete for the classes to be valid.
# The only residual nulls are the MNAR clinical numerics (diameter_1, fitspatrick), flagged above.
na = g.isna().sum(); na = na[na > 0]
print('Residual nulls in the cleaned table:', dict(na) if len(na) else 'none')
print('All residual nulls sit in non-biopsied rows:',
      bool((g.loc[g['diameter_1'].isna(), 'biopsed'].astype(str) == '0').all()))
assert all(g[q].isna().sum() == 0 for q in gen_qi), 'a generalised QI has nulls'
print('Generalised QIs complete (nulls):', {q: int(g[q].isna().sum()) for q in gen_qi})

In [ ]:
# Step 3.3b - Score k in {2,3,5,10}: records retained (raw vs generalised QIs), min class size, and a
# combined utility-privacy score = retained% x (1 - 1/k). Verify every surviving class has size >= k.
pr = g.groupby([g[q].astype(str) for q in gen_qi])[gen_qi[0]].transform('size')        # generalised classes
pr_raw = dp.groupby([dp[q].astype(str) for q in quasi])[quasi[0]].transform('size')    # raw classes
def report(per, k):
    kept=per[per>=k]
    return {'k':k,'retained_raw_%':round((pr_raw>=k).mean()*100,1),'retained_%':round((per>=k).mean()*100,1),
            'min_class_after':int(kept.min()) if len(kept) else 0,
            'utility_privacy':round((per>=k).mean()*100*(1-1/k),1)}
trade=pd.DataFrame([report(pr,k) for k in [2,3,5,10]])
print(trade.to_string(index=False))
assert all(trade['min_class_after']>=trade['k']), 'verification failed'   # n >= k in every kept class

In [ ]:
# Step 3.4 - Pick the operating k (max utility-privacy score) and build the final release:
# generalised QIs replace the raw ones, direct identifiers removed, classes < k suppressed.
best = trade.loc[trade['utility_privacy'].idxmax()]
K = int(best['k'])
print('Best configuration: k =', K, '(utility_privacy =', best['utility_privacy'], ')')
raw_qi   = ['age', 'city', 'postal_code', 'hospital_id', 'background_father', 'background_mother']
released = g.drop(columns=['dni', 'patient_id', 'lesion_id', 'img_id'] + raw_qi)
cls   = released.groupby([released[q].astype(str) for q in gen_qi])[gen_qi[0]].transform('size')
kanon = released[cls >= K].copy()                              # suppress records in classes smaller than k
final = kanon.groupby([kanon[q].astype(str) for q in gen_qi]).size()
print(f'Before k-anonymity (cleaned; DIs + raw QIs removed): {released.shape}')
print(f'After  k-anonymity (k={K}): {kanon.shape}  '
      f'-> {len(released)-len(kanon)} records suppressed ({(1-len(kanon)/len(released))*100:.1f}%)')
print('Verification | min class =', int(final.min()), '(>= k:', bool(final.min()>=K), ')')
print('QIs in the release (generalised only):', gen_qi)
assert final.min() >= K and kanon[gen_qi].isna().sum().sum() == 0, 'verification failed'   # zero outliers

In [ ]:
# Plot the privacy-utility trade-off: records retained on raw vs generalised QIs, per k.
ks=trade['k'].tolist(); x=np.arange(len(ks)); wbar=0.38
fig,ax=plt.subplots(figsize=(6.5,3.6))
ax.bar(x-wbar/2, trade['retained_raw_%'], wbar, label='Raw QI', color='#E74C3C')
ax.bar(x+wbar/2, trade['retained_%'], wbar, label='Generalised QI', color='#2471A3')
for i,(a,b) in enumerate(zip(trade['retained_raw_%'],trade['retained_%'])):
    ax.text(i-wbar/2,a+1,f'{a:.0f}',ha='center',fontsize=7); ax.text(i+wbar/2,b+1,f'{b:.0f}',ha='center',fontsize=7)
ax.set_xticks(x); ax.set_xticklabels([f'k={k}' for k in ks]); ax.set_ylabel('% records retained'); ax.set_ylim(0,108)
ax.legend(frameon=False,fontsize=8.5); ax.set_title('Privacy-utility trade-off: records retained vs k',fontsize=10,fontweight='bold')
plt.tight_layout(); plt.savefig(FIGURES_DIR/'fig_priv_tradeoff.png',bbox_inches='tight'); plt.show()

In [ ]:
# Per-class retention at the chosen k: share of each diagnosis surviving suppression (fairness check).
kept_ids = set(g.loc[pr >= K, 'patient_id'])
base = df['diagnostic'].value_counts(); kept = df[df['patient_id'].isin(kept_ids)]['diagnostic'].value_counts()
fair = pd.DataFrame({'total': base, 'kept': kept}).fillna(0).astype(int)
fair['retained_%'] = (fair['kept'] / fair['total'] * 100).round(1)
print(f'Per-class retention at k={K}:')
print(fair.sort_values('total', ascending=False).to_string())